# AG952 | Workshop 9 — BrewDog: From Punk Pioneer to Administration

**Textual Analytics for Accounting and Finance**  
University of Strathclyde Business School

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/materials/week09/notebooks/AG952_W9_BrewDog_Transformer_Student.ipynb)

---

## Overview

In this workshop you act as a **financial news analyst** examining BrewDog plc's media coverage arc from 2010 to 2025 — from its founding as a craft-beer disruptor through crowdfunding innovation, rapid international expansion, reputational collapse, and eventual entry into administration.

The corpus consists of **1,011 real articles** drawn from UK national newspapers (The Sun, The Times, the Financial Times, the Daily Mail, and others) via Factiva, covering the period 2010–2026.

The workshop is structured in two parts:

**Part 1 — Traditional NLP methods** (35 min)  
Apply LDA topic modelling and dictionary-based sentiment analysis to the BrewDog news corpus. Make explicit methodological choices (pre-processing pipeline, number of topics, sentiment dictionary) and evaluate their consequences. CP3 includes era-stratified LDA and a Structural Topic Modelling (STM) approximation to explore how BrewDog's narrative themes shifted across four distinct periods.

**Part 2 — Transformer methods** (20 min)  
Explore BERT-based tokenisation, contextual embeddings and attention. Compare domain-generic (DistilBERT-SST2) and finance-tuned (FinBERT) models. Evaluate method agreement and the accuracy–interpretability tradeoff.

**Analyst's Note** (5 min)  
Synthesise your findings in a maximum 150-word analyst note and submit to the class Google Sheet for debrief.

---

| Section | Checkpoint | Estimated time |
|---|---|---|
| Setup | CP0 — Team registration | 2 min |
| Part 1 | CP1 — Load corpus & timeline | 5 min |
| Part 1 | CP2 — Pre-processing | 5 min |
| Part 1 | CP3 — LDA by era + STM-style analysis | 15 min |
| Part 1 | CP4 — Dictionary sentiment | 8 min |
| Part 2 | CP5 — Transformer walkthrough | 8 min |
| Part 2 | CP6 — Run transformer model | 8 min |
| Part 2 | CP7 — Head-to-head comparison | 4 min |
| Submission | CP8 — Analyst's note | 5 min |

> **Note:** CP6 downloads a pre-trained model (~420 MB on first run). Start this as soon as you reach Part 2 — inference on CPU takes 1–3 minutes.


In [ ]:
#@title Configuration — instructor sets these before the session
APPS_SCRIPT_URL = ""  # paste deployed Apps Script URL here
DATA_URL = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week09/data/brewdog_articles_factiva.csv"


In [ ]:
#@title Step 0 — Install and import dependencies (run this first, ~60s)
!pip install transformers torch wordcloud vaderSentiment scikit-learn nltk requests pandas numpy matplotlib seaborn ipywidgets -q

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import time
import warnings
import requests
import random
import collections
from IPython.display import display, clear_output, Markdown, HTML

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import ipywidgets as widgets

warnings.filterwarnings("ignore")

from wordcloud import WordCloud

# ---------------------------------------------------------------------------
# Session data dict — accumulates all choices for final submission
# ---------------------------------------------------------------------------
GSHEET_COLUMNS = [
    "team_name",
    "cp1_period",
    "cp2_normalisation",
    "cp2_stopwords",
    "cp2_remove_numbers",
    "cp2_lowercase",
    "cp3_n_topics",
    "cp3_topic_labels",
    "cp4_dictionary",
    "cp4_sentiment_2010_2014",
    "cp4_sentiment_2019_2025",
    "cp6_model",
    "cp6_finbert_accuracy",
    "cp6_distilbert_accuracy",
    "cp7_interpretability_choice",
    "cp8_analyst_note",
    "timestamp",
]
session_data = {col: None for col in GSHEET_COLUMNS}

# ---------------------------------------------------------------------------
# Validation helper — blocks progression if prior step not completed
# ---------------------------------------------------------------------------
def validate_checkpoint(required_globals, checkpoint_name):
    missing = [v for v in required_globals if globals().get(v) is None]
    if missing:
        raise RuntimeError(
            f"Complete {checkpoint_name} before running this cell. "
            f"Still needed: {', '.join(missing)}"
        )

print("Dependencies loaded. Run CP0 to register your team.")


## Checkpoint 0 — Team Registration


In [ ]:
#@title CP0 — Register your team
# Instructor pre-configures TEAM_ASSIGNMENT before the session.
# All teams analyse the same BrewDog corpus but make independent methodological choices.

TEAM_ASSIGNMENT = {
    "The Disclosure Desk":   "BrewDog",
    "Alpha Analysts":        "BrewDog",
    "The Write-Offs":        "BrewDog",
    "Narrative Risk":        "BrewDog",
    "Sentiment Squad":       "BrewDog",
    "The Fog Index":         "BrewDog",
    "Punk Analysts":         "BrewDog",
    "Bear Market Brewers":   "BrewDog",
    "Item 1A-OK":            "BrewDog",
    "The Quant Collective":  "BrewDog",
    "FinText Co.":           "BrewDog",
    "Signals & Noise":       "BrewDog",
}

_cp0_team_w = widgets.Dropdown(
    options=["-- select your team --"] + sorted(TEAM_ASSIGNMENT.keys()),
    description="Team:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="420px"),
)
_cp0_out = widgets.Output()
_cp0_btn = widgets.Button(
    description="Register team",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
)

def _run_cp0(b):
    global _cp0_done
    _cp0_out.clear_output(wait=True)
    with _cp0_out:
        if _cp0_team_w.value == "-- select your team --":
            print("Select your team name first.")
            return
        session_data["team_name"] = _cp0_team_w.value
        _cp0_done = True
        print(f"Registered: {_cp0_team_w.value}")
        print(f"Subject: BrewDog plc — news corpus analysis, 2010–2025")
        print(f"\nObjective: Examine how financial and general news coverage of BrewDog")
        print(f"evolved from its founding as a craft-beer disruptor through to its")
        print(f"entry into administration in 2025.")
        print(f"\nCP0 complete. Run CP1 to load the corpus.")

_cp0_btn.on_click(_run_cp0)
display(widgets.VBox([_cp0_team_w, _cp0_btn, _cp0_out]))


In [ ]:
#@title CP1 — Load corpus and explore the BrewDog news timeline

validate_checkpoint(["_cp0_done"], "CP0")

# Load data
try:
    articles = pd.read_csv(DATA_URL)
    print(
        f"Corpus loaded: {len(articles)} articles, "
        f"{articles['year'].nunique()} years "
        f"({int(articles['year'].min())}–{int(articles['year'].max())})"
    )
except Exception as e:
    print(f"Could not fetch from URL ({e}). Attempting local fallback...")
    articles = pd.read_csv("AG952/materials/week09/data/brewdog_articles_factiva.csv")

articles["date"] = pd.to_datetime(articles["date"], errors="coerce")
articles["year"] = articles["year"].astype(int)

# Choose analysis period
_cp1_period_w = widgets.Dropdown(
    options=[
        "Full timeline (2010–2025)",
        "Early growth (2010–2015)",
        "Expansion era (2015–2019)",
        "Controversy & crisis (2019–2025)",
    ],
    description="Focus period:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px"),
)
_cp1_out = widgets.Output()
_cp1_btn = widgets.Button(
    description="Load corpus",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
)

def _run_cp1(b):
    global _cp1_done, working_articles, _period_label
    _cp1_out.clear_output(wait=True)
    with _cp1_out:
        period = _cp1_period_w.value
        session_data["cp1_period"] = period
        if "2010–2015" in period:
            working_articles = articles[articles["year"] <= 2015].copy()
        elif "2015–2019" in period:
            working_articles = articles[
                (articles["year"] >= 2015) & (articles["year"] <= 2019)
            ].copy()
        elif "2019–2025" in period:
            working_articles = articles[articles["year"] >= 2019].copy()
        else:
            working_articles = articles.copy()
        _period_label = period
        _cp1_done = True

        print(
            f"Working corpus: {len(working_articles)} articles "
            f"({int(working_articles['year'].min())}–{int(working_articles['year'].max())})"
        )
        print(f"\nTop newspapers:")
        for paper, n in working_articles["newspaper"].value_counts().head(8).items():
            print(f"  {paper:<35} {n:>4} articles")

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        # Chart 1: articles by year, colour-coded by era
        yr_counts = working_articles.groupby("year").size()
        colors = [
            "#ef4444" if y >= 2022 else
            ("#f97316" if y >= 2019 else
             ("#f59e0b" if y >= 2015 else "#22c55e"))
            for y in yr_counts.index
        ]
        axes[0].bar(
            yr_counts.index, yr_counts.values,
            color=colors, edgecolor="white", linewidth=0.5,
        )
        axes[0].set_title(
            "Articles by year\n(green=growth, amber=scaling, orange=controversy, red=crisis)"
        )
        axes[0].set_xlabel("Year")
        axes[0].set_ylabel("Article count")
        axes[0].tick_params(axis="x", rotation=45)

        # Chart 2: articles by newspaper (top 8)
        top_papers = working_articles["newspaper"].value_counts().head(8)
        paper_colors = plt.cm.tab10(range(len(top_papers)))
        axes[1].barh(
            top_papers.index[::-1], top_papers.values[::-1],
            color=paper_colors[::-1], edgecolor="white", linewidth=0.5,
        )
        axes[1].set_title("Articles by newspaper (top 8)")
        axes[1].set_xlabel("Article count")
        axes[1].tick_params(axis="y", labelsize=8)
        plt.tight_layout()
        plt.show()

        print("\nCP1 complete. Note the volume spike in 2021 (culture scandal) and")
        print("sustained coverage through 2022–2024 (financial distress).")
        print("Run the CP1 Feedback cell, then proceed to CP2.")

_cp1_btn.on_click(_run_cp1)
display(widgets.VBox([_cp1_period_w, _cp1_btn, _cp1_out]))


In [ ]:
#@title CP1 — Feedback (run after clicking Load corpus)
validate_checkpoint(["_cp1_done"], "CP1")
display(Markdown(
    "### CP1 Feedback — The BrewDog Story: From Punk Pioneer to Administration\n\n"
    "BrewDog was founded in Fraserburgh, Scotland in 2007 by James Watt and Martin Dickie. "
    "Its **Equity for Punks** crowdfunding model — which ultimately raised over £75m from more than 200,000 small "
    "investors — was widely covered as a disruptive innovation in both the craft beer and corporate finance press.\n\n"
    "**The corpus you have just loaded** contains 1,011 articles drawn from UK national newspapers — predominantly "
    "The Sun, The Times, the Financial Times, the Daily Mail, and The Guardian — covering BrewDog from 2010 to early "
    "2026. Note the source mix: tabloids (The Sun, Daily Mail) cover BrewDog primarily through a consumer and "
    "controversy lens; financial press (FT, FT.com) focus on investment, accounts, and financial distress. "
    "This source diversity is not neutral — it shapes what the corpus 'knows' about BrewDog.\n\n"
    "**Three distinct phases** are visible in the volume chart:\n\n"
    "**2010–2014 — The growth narrative.** Coverage focused on product innovation, anti-establishment brand "
    "positioning, and the mechanics of equity crowdfunding as a novel financing instrument. Article counts "
    "were low but tone was broadly celebratory.\n\n"
    "**2015–2018 — Complexity and scale.** A £213m minority investment by TSG Consumer Partners in 2017 attracted "
    "scrutiny about whether BrewDog had abandoned its punk credentials. Coverage broadened from beer trade to "
    "national business press.\n\n"
    "**2019–2025 — Reputational and financial collapse.** An open letter signed by 61 former employees in June 2021 "
    "alleging a 'culture of fear' triggered the largest sustained coverage spike in the corpus. Declining revenues, "
    "going-concern audit qualifications, bar closures and the eventual entry into administration in 2025 completed "
    "the arc — and explain the sustained red bars in your chart.\n\n"
    "> **Your analytical task:** Use the methods in this workshop to trace how this narrative is captured — and "
    "sometimes missed — by computational text analysis."
))
print("Proceed to CP2.")


## Part 1 — Traditional NLP Methods


In [ ]:
#@title CP2 — Pre-processing choices

validate_checkpoint(["_cp1_done"], "CP1")

_cp2_norm_w = widgets.Dropdown(
    options=["None", "Porter Stemmer", "Lemmatisation (WordNetLemmatizer)"],
    value="Lemmatisation (WordNetLemmatizer)",
    description="Normalisation:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="460px"),
)
_cp2_sw_w = widgets.Dropdown(
    options=[
        "NLTK standard",
        "Finance-adjusted (retain modals & negation)",
        "None",
    ],
    value="Finance-adjusted (retain modals & negation)",
    description="Stop words:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="460px"),
)
_cp2_num_w = widgets.RadioButtons(
    options=["Yes", "No"],
    value="Yes",
    description="Remove numbers:",
    style={"description_width": "180px"},
)
_cp2_lower_w = widgets.RadioButtons(
    options=["Yes", "No"],
    value="Yes",
    description="Lowercase:",
    style={"description_width": "180px"},
)
_cp2_out = widgets.Output()
_cp2_btn = widgets.Button(
    description="Apply pipeline",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
)

MODALS = {"will", "may", "might", "could", "should", "would", "must", "shall", "can"}
NEGATION = {"not", "no", "nor", "never", "neither", "without", "cannot"}


def _apply_pipeline(text):
    t = str(text)
    if _cp2_lower_w.value == "Yes":
        t = t.lower()
    if _cp2_norm_w.value == "Lemmatisation (WordNetLemmatizer)":
        lem = WordNetLemmatizer()
        tokens = [lem.lemmatize(w) for w in re.findall(r"[a-zA-Z]+", t)]
    elif _cp2_norm_w.value == "Porter Stemmer":
        stem = PorterStemmer()
        tokens = [stem.stem(w) for w in re.findall(r"[a-zA-Z]+", t)]
    else:
        tokens = re.findall(r"[a-zA-Z]+", t)
    if _cp2_num_w.value == "Yes":
        tokens = [w for w in tokens if w.isalpha()]
    sw = set(stopwords.words("english"))
    if _cp2_sw_w.value == "Finance-adjusted (retain modals & negation)":
        sw = sw - MODALS - NEGATION
    elif _cp2_sw_w.value == "None":
        sw = set()
    tokens = [w for w in tokens if w.lower() not in sw]
    return tokens


def _run_cp2(b):
    global _cp2_done, processed_docs, vocab_size
    _cp2_out.clear_output(wait=True)
    with _cp2_out:
        session_data.update(
            {
                "cp2_normalisation": _cp2_norm_w.value,
                "cp2_stopwords": _cp2_sw_w.value,
                "cp2_remove_numbers": _cp2_num_w.value,
                "cp2_lowercase": _cp2_lower_w.value,
            }
        )
        working_articles["tokens"] = working_articles["text"].apply(_apply_pipeline)
        working_articles["token_str"] = working_articles["tokens"].apply(" ".join)
        processed_docs = working_articles["token_str"].tolist()
        all_tokens = [t for toks in working_articles["tokens"] for t in toks]
        vocab_size = len(set(all_tokens))

        # Word cloud
        freq = collections.Counter(all_tokens)
        wc = WordCloud(
            width=800, height=300, background_color="white",
            colormap="Blues", max_words=100,
        ).generate_from_frequencies(freq)
        fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
        axes[0].imshow(wc, interpolation="bilinear")
        axes[0].axis("off")
        axes[0].set_title("Top terms across corpus")

        # Tokens per doc distribution
        doc_lengths = working_articles["tokens"].apply(len)
        axes[1].hist(doc_lengths, bins=25, color="#4f46e5", edgecolor="white", alpha=0.85)
        axes[1].set_title("Token count per article")
        axes[1].set_xlabel("Tokens")
        axes[1].set_ylabel("Articles")
        plt.tight_layout()
        plt.show()

        print(f"Pipeline applied. Vocabulary size: {vocab_size:,} unique tokens")
        print(
            f"Mean tokens per article: {doc_lengths.mean():.0f} "
            f"(min {doc_lengths.min()}, max {doc_lengths.max()})"
        )
        sample = working_articles["tokens"].iloc[0][:20]
        print(f"\nSample (first 20 tokens of article 0): {sample}")
        _cp2_done = True
        print("\nCP2 complete. Run the CP2 Feedback cell, then proceed to CP3.")


_cp2_btn.on_click(_run_cp2)
display(widgets.VBox([_cp2_norm_w, _cp2_sw_w, _cp2_num_w, _cp2_lower_w, _cp2_btn, _cp2_out]))


In [ ]:
#@title CP2 — Feedback

validate_checkpoint(["_cp2_done"], "CP2")

_fb = {
    "None": (
        "**No normalisation** preserves surface forms. This maximises transparency but inflates "
        "vocabulary with variants (e.g. *fund*, *funding*, *funded* treated as separate types). "
        "For a topic model, this increases sparsity without adding information."
    ),
    "Porter Stemmer": (
        "**Porter Stemming** aggressively truncates suffixes (*controversy* \u2192 *controversi*). "
        "It reduces vocabulary size efficiently but produces non-words that complicate human "
        "interpretation of topic word-lists \u2014 a significant cost when you need students to "
        "label topics meaningfully."
    ),
    "Lemmatisation (WordNetLemmatizer)": (
        "**Lemmatisation** reduces to dictionary base forms (*employees* \u2192 *employee*, "
        "*declining* \u2192 *decline*). It achieves vocabulary compression while preserving "
        "interpretable roots. For topic modelling of news text, this is generally the strongest choice."
    ),
}
_sw_fb = {
    "NLTK standard": (
        "The **standard NLTK list** removes function words efficiently. However, it also removes "
        "modals (*may*, *could*, *will*) and negation (*not*, *no*) \u2014 words that carry "
        "critical sentiment signal in financial coverage."
    ),
    "Finance-adjusted (retain modals & negation)": (
        "The **finance-adjusted list** retains modal verbs and negation terms. This is the better "
        "choice for sentiment analysis because it preserves uncertainty language (*may face "
        "insolvency*) and negation (*did not recover*) that dictionaries and classifiers depend on."
    ),
    "None": (
        "**No stop-word removal** leaves function words in the vocabulary. This adds noise to "
        "topic models and dilutes sentiment signal. It is rarely the right choice for analysis "
        "of news text."
    ),
}

norm_choice = session_data.get("cp2_normalisation", "")
sw_choice = session_data.get("cp2_stopwords", "")
norm_text = _fb.get(norm_choice, "")
sw_text = _sw_fb.get(sw_choice, "")

display(Markdown(
    f"### CP2 Feedback \u2014 Pre-processing choices and their consequences\n\n"
    f"{norm_text}\n\n"
    f"{sw_text}\n\n"
    "**Note on context:** These choices were originally designed for formal financial disclosures "
    "(10-K filings). News text has shorter documents, more colloquial language, and higher topical "
    "diversity. The finance-adjusted stop-word list retains its value here because modals and "
    "negation still carry critical sentiment signal in financial journalism."
))
print("Proceed to CP3.")


In [ ]:
#@title CP3 — Topic modelling: LDA by narrative era + STM-style temporal analysis

validate_checkpoint(["_cp2_done"], "CP2")

# BrewDog's four narrative eras (based on known arc)
ERAS = {
    "Growth (2010–2014)":      (2010, 2014),
    "Scaling (2015–2018)":     (2015, 2018),
    "Controversy (2019–2021)": (2019, 2021),
    "Crisis (2022–2025)":      (2022, 2025),
}

_cp3_ntopics_w = widgets.IntSlider(
    value=6, min=3, max=12, step=1,
    description="Topics (k):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
    continuous_update=False,
)
_cp3_out = widgets.Output()
_cp3_run_btn = widgets.Button(
    description="Run LDA",
    button_style="primary",
    layout=widgets.Layout(width="150px"),
)
_cp3_label_boxes = []
_cp3_label_box_container = widgets.VBox([])
_cp3_label_btn = widgets.Button(
    description="Save topic labels",
    button_style="success",
    layout=widgets.Layout(width="180px"),
    disabled=True,
)
_cp3_era_btn = widgets.Button(
    description="Compare eras",
    button_style="info",
    layout=widgets.Layout(width="160px"),
    disabled=True,
)
_cp3_stm_btn = widgets.Button(
    description="STM-style analysis",
    button_style="warning",
    layout=widgets.Layout(width="190px"),
    disabled=True,
)


# ── Step 1: Run global LDA ─────────────────────────────────────────────────
def _run_cp3_lda(b):
    global _lda_model, _lda_vectorizer, _doc_topics, _cp3_run_done
    _cp3_out.clear_output(wait=True)
    with _cp3_out:
        k = _cp3_ntopics_w.value
        session_data["cp3_n_topics"] = k
        print(f"Fitting LDA with k={k} topics on {len(processed_docs)} documents...")
        _lda_vectorizer = CountVectorizer(max_features=3000, min_df=2, max_df=0.85)
        dtm = _lda_vectorizer.fit_transform(processed_docs)
        _lda_model = LatentDirichletAllocation(
            n_components=k, random_state=42, learning_method="batch", max_iter=30,
        )
        _lda_model.fit(dtm)
        _doc_topics = _lda_model.transform(dtm)
        working_articles["dominant_topic"] = _doc_topics.argmax(axis=1)

        vocab = _lda_vectorizer.get_feature_names_out()
        print(f"\nTop 10 words per topic (global model — full corpus):\n")
        for i, comp in enumerate(_lda_model.components_):
            top_words = [vocab[j] for j in comp.argsort()[-10:][::-1]]
            print(f"  Topic {i}: {', '.join(top_words)}")

        print("\nEnter a short label for each topic below, then click Save topic labels.")
        _cp3_run_done = True
        _cp3_label_btn.disabled = False

        boxes = []
        for i in range(k):
            comp = _lda_model.components_[i]
            top_w = [vocab[j] for j in comp.argsort()[-5:][::-1]]
            box = widgets.Text(
                value=f"Topic {i}",
                description=f"Topic {i} ({', '.join(top_w[:3])}):",
                style={"description_width": "280px"},
                layout=widgets.Layout(width="600px"),
            )
            boxes.append(box)
        _cp3_label_boxes[:] = boxes
        _cp3_label_box_container.children = (
            [widgets.HTML("<b>Enter topic labels (read the top words above first):</b>")] + boxes
        )


# ── Step 2: Save labels ────────────────────────────────────────────────────
def _save_labels(b):
    global _cp3_done
    _cp3_out.clear_output(wait=True)
    with _cp3_out:
        labels = [
            w.value.strip() or f"Topic {i}"
            for i, w in enumerate(_cp3_label_boxes)
        ]
        label_map = {i: lbl for i, lbl in enumerate(labels)}
        working_articles["topic_label"] = working_articles["dominant_topic"].map(label_map)
        session_data["cp3_topic_labels"] = "; ".join(f"{i}:{lbl}" for i, lbl in label_map.items())

        print("Topic labels saved:")
        for i, lbl in label_map.items():
            print(f"  Topic {i} → {lbl}")
        _cp3_done = True
        _cp3_era_btn.disabled = False
        _cp3_stm_btn.disabled = False
        print("\nNext steps:")
        print("  'Compare eras'      — see how topics differ across BrewDog's four narrative periods")
        print("  'STM-style analysis' — see how topic prevalence changes over time")


# ── Step 3: Era comparison ─────────────────────────────────────────────────
def _run_era_comparison(b):
    with _cp3_out:
        print("\n" + "─" * 60)
        print("Era comparison: LDA fitted separately per narrative period")
        print("─" * 60)

        era_k = max(3, min(_cp3_ntopics_w.value, 4))
        era_top_terms = {}

        for era_name, (yr_start, yr_end) in ERAS.items():
            era_df = working_articles[
                (working_articles["year"] >= yr_start) &
                (working_articles["year"] <= yr_end)
            ]
            era_texts = era_df["token_str"].tolist()

            if len(era_texts) < 5:
                era_top_terms[era_name] = []
                print(f"  {era_name}: insufficient articles ({len(era_texts)}) — skipped.")
                continue

            vect_era = CountVectorizer(max_features=1500, min_df=1, max_df=0.95)
            dtm_era = vect_era.fit_transform(era_texts)
            lda_era = LatentDirichletAllocation(
                n_components=era_k, random_state=42, max_iter=25,
            )
            lda_era.fit(dtm_era)
            vocab_era = vect_era.get_feature_names_out()

            # Aggregate scores across all topics, deduplicate
            term_scores = {}
            for comp in lda_era.components_:
                for j in comp.argsort()[-15:][::-1]:
                    w = vocab_era[j]
                    term_scores[w] = term_scores.get(w, 0) + comp[j]
            top_terms = [w for w, _ in sorted(term_scores.items(), key=lambda x: -x[1])[:12]]
            era_top_terms[era_name] = top_terms

        # Print as a comparison table
        print()
        header = f"{'#':<4}" + "".join(f"{e:<26}" for e in ERAS)
        print(header)
        print("─" * len(header))
        n_rows = max((len(v) for v in era_top_terms.values()), default=0)
        for i in range(n_rows):
            row = f"{i+1:<4}"
            for era in ERAS:
                terms = era_top_terms.get(era, [])
                row += f"{terms[i] if i < len(terms) else '':<26}"
            print(row)

        # Heatmap: raw term frequency per era for the top terms found
        all_terms = []
        for terms in era_top_terms.values():
            for t in terms[:8]:
                if t not in all_terms:
                    all_terms.append(t)
        all_terms = all_terms[:24]

        heat_rows, era_labels = [], []
        for era_name, (yr_start, yr_end) in ERAS.items():
            era_df = working_articles[
                (working_articles["year"] >= yr_start) &
                (working_articles["year"] <= yr_end)
            ]
            era_text = " ".join(era_df["token_str"].tolist())
            tokens = era_text.split()
            total = max(len(tokens), 1)
            heat_rows.append([tokens.count(t) / total * 1000 for t in all_terms])
            era_labels.append(era_name)

        heat_df = pd.DataFrame(heat_rows, index=era_labels, columns=all_terms)

        fig, ax = plt.subplots(figsize=(14, 3.5))
        sns.heatmap(
            heat_df, ax=ax, cmap="YlOrRd",
            annot=True, fmt=".1f", linewidths=0.4,
            cbar_kws={"label": "Occurrences per 1,000 tokens"},
        )
        ax.set_title(
            "Key LDA terms by narrative era  (frequency per 1,000 tokens)\n"
            "Bright cells = term was disproportionately prominent in that era"
        )
        plt.xticks(rotation=45, ha="right", fontsize=8)
        plt.tight_layout()
        plt.show()

        print("\nLook for terms concentrated in a single era — those are era-defining topics.")
        print("Terms that appear across all eras are corpus-wide background vocabulary.")


# ── Step 4: STM-style temporal analysis ───────────────────────────────────
def _run_stm_style(b):
    with _cp3_out:
        print("\n" + "─" * 60)
        print("STM-style analysis: topic prevalence over time")
        print("─" * 60)

        label_map_rev = {f"Topic {i}": w.value.strip() or f"Topic {i}"
                         for i, w in enumerate(_cp3_label_boxes)}
        topic_cols = [f"Topic {i}" for i in range(_lda_model.n_components)]
        props_df = pd.DataFrame(_doc_topics, columns=topic_cols)
        props_df["year"] = working_articles["year"].values
        yr_props = props_df.groupby("year")[topic_cols].mean()
        yr_props.columns = [label_map_rev.get(c, c) for c in yr_props.columns]

        # OLS slope per topic (proportion ~ year)
        years_arr = yr_props.index.values.astype(float)
        years_c = years_arr - years_arr.mean()
        print("\nTopic prevalence trend (OLS: proportion ~ year):\n")
        slopes = {}
        for col in yr_props.columns:
            slope, _ = np.polyfit(years_c, yr_props[col].values, 1)
            direction = "↑ rising" if slope > 0.005 else ("↓ declining" if slope < -0.005 else "→ stable")
            slopes[col] = slope
            print(f"  {col:<32} slope = {slope:+.5f}   {direction}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        yr_props.plot(ax=axes[0], marker="o", colormap="tab10", linewidth=1.8)
        # Shade narrative eras
        yr_min, yr_max = years_arr.min(), years_arr.max()
        for shade_start, shade_end, color, label in [
            (2010, 2014.5, "#bbf7d0", "Growth"),
            (2014.5, 2018.5, "#fef08a", "Scaling"),
            (2018.5, 2021.5, "#fecaca", "Controversy"),
            (2021.5, 2025.5, "#fca5a5", "Crisis"),
        ]:
            if shade_end >= yr_min and shade_start <= yr_max:
                axes[0].axvspan(
                    max(shade_start, yr_min), min(shade_end, yr_max + 0.5),
                    alpha=0.18, color=color, zorder=0,
                )
        axes[0].set_title("Topic prevalence by year\n(mean document-topic proportion)")
        axes[0].set_xlabel("Year")
        axes[0].set_ylabel("Mean proportion")
        axes[0].legend(title="Topic", fontsize=7, bbox_to_anchor=(1.02, 1))

        slope_s = pd.Series(slopes).sort_values()
        bar_colors = ["#22c55e" if v > 0 else "#ef4444" for v in slope_s]
        slope_s.plot(kind="barh", ax=axes[1], color=bar_colors, edgecolor="white")
        axes[1].axvline(0, color="grey", linewidth=0.8)
        axes[1].set_title("Topic trend direction\n(OLS slope: positive = rising over time)")
        axes[1].set_xlabel("Slope coefficient (proportion ~ year)")
        plt.tight_layout()
        plt.show()

        display(Markdown(
            "---\n"
            "### What is Structural Topic Modelling (STM)?\n\n"
            "The analysis above is a manual approximation of what **Structural Topic Modelling (STM)** "
            "formalises statistically (Roberts, Stewart & Tingley, 2019).\n\n"
            "**What we just did — in four steps:**\n"
            "1. Fitted LDA on the full corpus to discover topics\n"
            "2. Extracted each document's topic proportion vector\n"
            "3. Grouped proportions by year to see how topic weight shifts over time\n"
            "4. Estimated a trend slope using OLS regression (proportion ~ year)\n\n"
            "**What STM does differently:** STM incorporates document-level metadata — such as `year`, "
            "`era`, or `firm type` — *inside* the model during estimation. Instead of discovering topics "
            "and *then* checking how year relates to them, STM uses year as a prevalence covariate when "
            "learning what the topics are. This means the topics themselves are already shaped by the "
            "temporal metadata, producing more temporally coherent themes than standard LDA.\n\n"
            "STM also supports **content covariates**: metadata can influence which *words* appear in a "
            "topic across groups. For example, an STM with `content ~ era` might reveal that the *Finance* "
            "topic uses different vocabulary in the Growth era (*crowdfunding*, *equity*, *punk*) versus "
            "the Crisis era (*administration*, *restructuring*, *going-concern*).\n\n"
            "> **Why we use the manual approximation here:** Python's scikit-learn LDA does not support "
            "STM natively. The canonical implementation is the R package `stm` (Roberts et al., 2019). "
            "The manual approach gives you the same interpretive output; the R version provides more "
            "rigorous statistical estimates and confidence intervals around covariate effects.\n\n"
            "**Key reference:** Roberts, M.E., Stewart, B.M., & Tingley, D. (2019). stm: An R package "
            "for structural topic models. *Journal of Statistical Software*, 91(2), 1–40."
        ))

        print("\nCP3 complete. Run the CP3 Feedback cell, then proceed to CP4.")


_cp3_run_btn.on_click(_run_cp3_lda)
_cp3_label_btn.on_click(_save_labels)
_cp3_era_btn.on_click(_run_era_comparison)
_cp3_stm_btn.on_click(_run_stm_style)

display(widgets.VBox([
    _cp3_ntopics_w,
    widgets.HBox([_cp3_run_btn, _cp3_label_btn, _cp3_era_btn, _cp3_stm_btn]),
    _cp3_label_box_container,
    _cp3_out,
]))

In [ ]:
#@title CP3 — Feedback

validate_checkpoint(["_cp3_done"], "CP3")

_k = session_data.get("cp3_n_topics", "?")
display(Markdown(
    f"### CP3 Feedback — Topic modelling: decisions, era comparison, and temporal structure\n\n"
    f"**Number of topics (k = {_k}).**  \n"
    "There is no statistically correct value of k for LDA; it is always a researcher decision. "
    "Too few topics conflates the *craft beer* narrative with *financial distress* simply because "
    "both appear in later-year articles. Too many topics fragments coherent themes into near-identical "
    "sub-topics and makes labelling unreliable. For a corpus of ~150 articles spanning 15 years, "
    "k between 5 and 8 typically balances coherence and coverage.\n\n"
    "**Why run LDA per era?**  \n"
    "A single LDA on the full corpus assumes that the same set of topics is present throughout — "
    "it can only learn that *administration* co-occurs with *losses* and *employees* if those words "
    "appear together in enough documents. When the corpus spans a dramatic narrative shift (craft beer "
    "success → reputational collapse → financial failure), a single global model blends vocabulary "
    "from incompatible periods. Fitting separate models per era allows each era's distinctive "
    "vocabulary to dominate without being diluted by unrelated periods.\n\n"
    "**Reading the era heatmap.**  \n"
    "Cells that are bright for one era and near-zero for others identify era-defining vocabulary. "
    "Terms that are uniformly present across all eras are corpus-wide background noise — words like "
    "*beer* or *company* tell you little about the narrative arc. The most analytically useful "
    "cells are those that show a clear temporal concentration.\n\n"
    "**STM and the value of metadata.**  \n"
    "The OLS slope table shows which topics are rising and which are declining over the 15-year "
    "period. A negative slope on a *Growth & Crowdfunding* topic and a positive slope on a "
    "*Controversy & Culture* topic would be consistent with what we know about BrewDog's arc. "
    "Structural Topic Modelling formalises this relationship by letting the year covariate shape "
    "topic discovery, rather than applying it retrospectively. The key interpretive advantage of "
    "STM in a case like BrewDog is that it can distinguish between topics that *always existed* "
    "but became more prominent over time, versus topics that genuinely *emerged* as new themes.\n\n"
    "**Ethical and methodological considerations:**\n\n"
    "1. **Topic labels are interpretations, not facts.** Two analysts given the same word list may "
    "assign different labels. Always report the top words alongside the label when presenting "
    "topic model results — the label is your interpretation of the words, not a ground truth.\n\n"
    "2. **Era boundaries are researcher choices.** The four eras used here (Growth / Scaling / "
    "Controversy / Crisis) were defined by knowledge of the BrewDog narrative. In a real analysis, "
    "you would not always have this prior knowledge. Consider how you would identify meaningful "
    "breakpoints from the data alone.\n\n"
    "3. **Training data coverage is uneven.** The 2021 open letter from 61 former employees "
    "generated a disproportionate volume of coverage in a short period. LDA will weight this "
    "burst of vocabulary heavily, potentially overstating the prominence of the workplace "
    "controversy topic relative to equally significant but less-covered issues.\n\n"
    "> **Question to consider:** If you applied the global LDA model to each era's documents "
    "separately (without re-fitting), would the dominant topics differ from those found by the "
    "era-specific models? What does this tell you about the difference between applying and "
    "estimating a topic model?"
))
print("Proceed to CP4.")

In [ ]:
#@title CP4 — Dictionary sentiment analysis

validate_checkpoint(["_cp3_done"], "CP3")

# Load LM and Harvard IV word lists from GitHub
_LM_NEG_URL = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/lm_negative.txt"
_LM_POS_URL = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/lm_positive.txt"
_HIV4_NEG_URL = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/harvard_iv4_negative.txt"
_HIV4_POS_URL = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/harvard_iv4_positive.txt"


def _load_wordlist(url):
    try:
        r = requests.get(url, timeout=10)
        return set(r.text.lower().split())
    except Exception:
        return set()


_cp4_dict_w = widgets.Dropdown(
    options=[
        "VADER (news-optimised)",
        "Loughran-McDonald (financial filings)",
        "Harvard IV-4 (general)",
        "All three (comparison mode)",
    ],
    value="All three (comparison mode)",
    description="Dictionary:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="500px"),
)
_cp4_out = widgets.Output()
_cp4_btn = widgets.Button(
    description="Run sentiment",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
)


def _run_cp4(b):
    global _cp4_done
    _cp4_out.clear_output(wait=True)
    with _cp4_out:
        print("Loading word lists...")
        lm_neg = _load_wordlist(_LM_NEG_URL)
        lm_pos = _load_wordlist(_LM_POS_URL)
        hiv4_neg = _load_wordlist(_HIV4_NEG_URL)
        hiv4_pos = _load_wordlist(_HIV4_POS_URL)
        vader = SentimentIntensityAnalyzer()
        session_data["cp4_dictionary"] = _cp4_dict_w.value

        def dict_score(tokens, pos_words, neg_words):
            p = sum(1 for t in tokens if t.lower() in pos_words)
            n = sum(1 for t in tokens if t.lower() in neg_words)
            return (p - n) / (p + n) if (p + n) > 0 else 0.0

        working_articles["vader_score"] = working_articles["text"].apply(
            lambda t: vader.polarity_scores(str(t))["compound"]
        )
        working_articles["lm_score"] = working_articles["tokens"].apply(
            lambda toks: dict_score(toks, lm_pos, lm_neg)
        )
        working_articles["hiv4_score"] = working_articles["tokens"].apply(
            lambda toks: dict_score(toks, hiv4_pos, hiv4_neg)
        )

        yr_mean = working_articles.groupby("year")[
            ["vader_score", "lm_score", "hiv4_score"]
        ].mean()
        session_data["cp4_sentiment_2010_2014"] = round(
            yr_mean[yr_mean.index <= 2014]["vader_score"].mean(), 4
        )
        session_data["cp4_sentiment_2019_2025"] = round(
            yr_mean[yr_mean.index >= 2019]["vader_score"].mean(), 4
        )

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        choice = _cp4_dict_w.value
        if choice == "All three (comparison mode)":
            for col, lbl, col_color in [
                ("vader_score", "VADER", "#4f46e5"),
                ("lm_score", "LM", "#f59e0b"),
                ("hiv4_score", "Harvard IV-4", "#10b981"),
            ]:
                axes[0].plot(yr_mean.index, yr_mean[col], marker="o", label=lbl, color=col_color)
        else:
            col = (
                "vader_score" if "VADER" in choice
                else ("lm_score" if "Loughran" in choice else "hiv4_score")
            )
            lbl = choice.split(" (")[0]
            axes[0].plot(yr_mean.index, yr_mean[col], marker="o", color="#4f46e5", label=lbl)

        axes[0].axhline(0, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
        axes[0].axvspan(
            2019, yr_mean.index.max() + 0.5,
            alpha=0.08, color="red", label="Controversy/crisis period",
        )
        axes[0].set_title("Mean sentiment score by year")
        axes[0].set_xlabel("Year")
        axes[0].set_ylabel("Sentiment score (−1 to +1)")
        axes[0].legend(fontsize=8)

        recent = working_articles[working_articles["year"] >= 2019]
        early = working_articles[working_articles["year"] <= 2014]
        axes[1].boxplot(
            [early["vader_score"].dropna(), recent["vader_score"].dropna()],
            labels=["2010–2014 (early)", "2019–2025 (crisis)"],
            patch_artist=True,
            boxprops=dict(facecolor="#e0f2fe"),
            medianprops=dict(color="#0ea5e9", lw=2),
        )
        axes[1].axhline(0, color="grey", linestyle="--", linewidth=0.8)
        axes[1].set_title("VADER: score distribution by period")
        axes[1].set_ylabel("VADER compound score")
        plt.tight_layout()
        plt.show()

        if choice == "All three (comparison mode)":
            print("\nDivergence between methods (2019–2025 mean scores):")
            crisis = working_articles[working_articles["year"] >= 2019]
            print(f"  VADER:        {crisis['vader_score'].mean():+.4f}")
            print(f"  LM:           {crisis['lm_score'].mean():+.4f}")
            print(f"  Harvard IV-4: {crisis['hiv4_score'].mean():+.4f}")
            print("\n  VADER is calibrated for news text; LM and Harvard IV-4 were designed")
            print("  for financial disclosures. Divergence between methods in the crisis")
            print("  period reveals dictionary domain mismatch — the subject of Part 2.")

        _cp4_done = True
        print("\nCP4 complete. Run the CP4 Feedback cell.")


_cp4_btn.on_click(_run_cp4)
display(widgets.VBox([_cp4_dict_w, _cp4_btn, _cp4_out]))


In [ ]:
#@title CP4 — Feedback

validate_checkpoint(["_cp4_done"], "CP4")

_early = session_data.get("cp4_sentiment_2010_2014")
_crisis = session_data.get("cp4_sentiment_2019_2025")
_early_str = f"{_early:+.4f}" if isinstance(_early, float) else str(_early)
_crisis_str = f"{_crisis:+.4f}" if isinstance(_crisis, float) else str(_crisis)

display(Markdown(
    f"### CP4 Feedback \u2014 Domain specificity and the dictionary problem\n\n"
    f"**Early sentiment ({_early_str}) vs crisis-period sentiment ({_crisis_str}).**\n\n"
    "The magnitude of the decline is real \u2014 but is the dictionary correctly attributing sentiment? "
    "Consider three failure modes:\n\n"
    "1. **Domain mismatch.** The Loughran-McDonald dictionary was constructed from SEC 10-K filings, "
    "where *liability*, *loss* and *default* are clearly negative. In news text, these words often appear "
    "in analytical contexts that are not inherently negative: *'BrewDog faces liability questions'* and "
    "*'BrewDog faces liability under contract law'* have very different valences, yet both score "
    "identically on LM.\n\n"
    "2. **Negation handling.** Unless you chose the finance-adjusted stop-word list, negation terms "
    "like *not* and *never* were stripped before scoring. This means *'revenues did not decline'* and "
    "*'revenues declined'* produce identical scores \u2014 a significant analytical error in coverage of "
    "financial distress.\n\n"
    "3. **VADER vs financial dictionaries.** VADER was trained on social media and news text, making "
    "it better calibrated for this corpus. However, it was not designed for financial language and will "
    "systematically under-weight domain-specific terms like *administration*, *going concern*, and "
    "*restructuring* that are strongly negative in context.\n\n"
    "> **The fundamental tension:** dictionaries are transparent and reproducible but brittle. In Part 2, "
    "you will test whether transformer models resolve these failure modes \u2014 and at what cost to "
    "interpretability."
))
print("Part 1 complete. Proceed to Part 2.")


## Part 2 — Transformer Methods

### What is a transformer?

A transformer is a neural network architecture that processes text as a sequence of tokens and builds a rich contextual representation of each token by attending to all other tokens in the sequence simultaneously. This is the mechanism that allows a transformer to distinguish *“the company managed its risk of administration”* (neutral/positive) from *“the company entered administration”* (negative) — the same word, *administration*, receives a fundamentally different vector representation depending on its context.

**BERT** (Bidirectional Encoder Representations from Transformers; Devlin et al., 2019) is the foundational pre-trained transformer. It was trained on 3.3 billion words using two objectives: masked language modelling (predict a hidden word from context) and next-sentence prediction. This pre-training gives BERT deep representations of general English. Fine-tuning adapts these representations to a specific task using a smaller labelled dataset.

### Learning objectives for Part 2

By the end of Part 2 you should be able to:

1. Explain what sub-word tokenisation is and why it matters for domain-specific text
2. Describe how contextual embeddings differ from bag-of-words representations
3. Interpret attention heatmaps as (partial) evidence of model reasoning
4. Compare the accuracy of a domain-generic model (DistilBERT-SST2) against a finance-tuned model (FinBERT) on the BrewDog corpus
5. Articulate the accuracy–interpretability tradeoff and its implications for regulated financial practice


In [ ]:
#@title CP5 — Transformer architecture: tokenisation, embeddings, and attention

validate_checkpoint(["_cp4_done"], "CP4")

from transformers import AutoTokenizer
import torch

_CP5_MODEL = "ProsusAI/finbert"
_cp5_out = widgets.Output()
_cp5_tok_btn = widgets.Button(
    description="A: Show tokenisation",
    button_style="primary",
    layout=widgets.Layout(width="230px"),
)
_cp5_emb_btn = widgets.Button(
    description="B: Show context sensitivity",
    button_style="primary",
    layout=widgets.Layout(width="230px"),
    disabled=True,
)
_cp5_att_btn = widgets.Button(
    description="C: Visualise attention",
    button_style="primary",
    layout=widgets.Layout(width="230px"),
    disabled=True,
)

_tokenizer_loaded = [None]


def _run_tok(b):
    _cp5_out.clear_output(wait=True)
    with _cp5_out:
        print("Loading FinBERT tokeniser (first run downloads ~420 MB)...")
        tok = AutoTokenizer.from_pretrained(_CP5_MODEL)
        _tokenizer_loaded[0] = tok
        _cp5_emb_btn.disabled = False

        example_sentences = [
            "BrewDog reported record revenue growth driven by international expansion.",
            "BrewDog entered administration after a sustained period of financial distress.",
        ]
        for sent in example_sentences:
            tokens = tok.tokenize(sent)
            ids = tok.encode(sent, add_special_tokens=True)
            print(f"\nSentence: '{sent}'")
            print(f"  Tokens ({len(tokens)}): {tokens}")
            print(f"  Token IDs: {ids}")
            print(f"  Note: [CLS] (id={ids[0]}) and [SEP] (id={ids[-1]}) are special tokens added by BERT.")
            print(f"  'administration' → {tok.tokenize('administration')} (single token; in-vocabulary)")
            print(f"  'BrewDog'        → {tok.tokenize('BrewDog')} (may split: sub-word tokenisation)")

        display(Markdown(
            "**What is sub-word tokenisation?**  \n"
            "BERT uses WordPiece tokenisation. Rare or domain-specific words are split into sub-word units "
            "that *were* seen during pre-training. For example, *BrewDog* may split into *Brew* + *##Dog*. "
            "The `##` prefix marks continuation of the previous token.\n\n"
            "**Why does this matter for financial text?**  \n"
            "Company names, financial instruments and industry jargon (e.g. *overcapitalisation*, *crowdfunding*) "
            "often split in ways that disrupt meaning. A model fine-tuned on financial text (like FinBERT) has "
            "seen these terms in context more often, so their sub-word representations are better calibrated.\n\n"
            "**The [CLS] token** is the classification token — the model's representation of the *whole sentence* "
            "is encoded here, and it is this vector that the sentiment classifier reads."
        ))


def _run_emb(b):
    with _cp5_out:
        display(Markdown(
            "---\n"
            "### B — Context sensitivity: why transformers outperform bag-of-words\n\n"
            "A critical limitation of dictionary methods (and LDA) is that they treat text as a "
            "**bag of words**: word order and context are discarded.\n\n"
            "Consider these two sentences from the BrewDog corpus:\n\n"
            "| Sentence | LM score | Human label |\n"
            "|---|---|---|\n"
            "| *'The risk of administration has increased materially'* | Negative (correct) | Negative |\n"
            "| *'The company has effectively managed the risk of administration'* | Negative (**wrong**) | Positive/Neutral |\n\n"
            "Both sentences contain *risk* and *administration* — both LM-negative words. The dictionary scores both negative.\n\n"
            "A transformer reads the full sentence as a sequence. The token *administration* in the second sentence "
            "attends to *managed* and *effectively*, receiving a contextual representation that reflects mitigation "
            "rather than occurrence.\n\n"
            "**This is the core claim of transformer superiority:** the same word in a different context receives a "
            "different numerical representation."
        ))
        _cp5_att_btn.disabled = False


def _run_att(b):
    global _cp5_done
    with _cp5_out:
        try:
            from transformers import AutoModelForSequenceClassification
            tok = _tokenizer_loaded[0]
            print("\nLoading FinBERT model for attention extraction...")
            model = AutoModelForSequenceClassification.from_pretrained(
                _CP5_MODEL, output_attentions=True
            )
            model.eval()

            text = (
                "BrewDog entered administration after a sustained period of "
                "financial losses and declining revenue."
            )
            inputs = tok(text, return_tensors="pt", truncation=True, max_length=64)
            with torch.no_grad():
                out = model(**inputs)
            att = out.attentions
            att_avg = (
                torch.stack([att[-1], att[-2]])
                .squeeze(1)
                .mean(dim=[0, 1])
                .numpy()
            )
            tokens_list = tok.convert_ids_to_tokens(inputs["input_ids"][0])

            fig, ax = plt.subplots(figsize=(9, 7))
            im = ax.imshow(att_avg, cmap="Blues", aspect="auto")
            ax.set_xticks(range(len(tokens_list)))
            ax.set_xticklabels(tokens_list, rotation=45, ha="right", fontsize=8)
            ax.set_yticks(range(len(tokens_list)))
            ax.set_yticklabels(tokens_list, fontsize=8)
            ax.set_title(
                f"Averaged attention (last 2 layers, all heads)\n'{text[:60]}...'"
            )
            plt.colorbar(im, ax=ax, fraction=0.03)
            plt.tight_layout()
            plt.show()

            print("\nEach cell (row i, col j) shows how much token i attends to token j.")
            print("Bright cells on the 'losses' and 'administration' rows indicate")
            print("those tokens draw heavily on surrounding context — consistent with")
            print("the model encoding their negative financial meaning from context.")
        except Exception as e:
            print(
                f"Attention extraction failed ({e}). "
                "This can occur on Colab CPU — continue to CP6."
            )
        _cp5_done = True
        print("\nCP5 complete. Proceed to CP6.")


_cp5_tok_btn.on_click(_run_tok)
_cp5_emb_btn.on_click(_run_emb)
_cp5_att_btn.on_click(_run_att)
display(
    widgets.VBox(
        [
            widgets.HBox([_cp5_tok_btn, _cp5_emb_btn, _cp5_att_btn]),
            _cp5_out,
        ]
    )
)


In [ ]:
#@title CP6 — Transformer sentiment: choose your model and run on the BrewDog corpus

validate_checkpoint(["_cp5_done"], "CP5")

_cp6_model_w = widgets.Dropdown(
    options=[
        "DistilBERT-SST2 (generic, fast — not finance-tuned)",
        "FinBERT (finance-tuned, slower)",
    ],
    value="FinBERT (finance-tuned, slower)",
    description="Model:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="520px"),
)
_cp6_sample_w = widgets.IntSlider(
    value=40, min=20, max=120, step=10,
    description="Sample size:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
    continuous_update=False,
)
_cp6_out = widgets.Output()
_cp6_btn = widgets.Button(
    description="Run transformer",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)


def _run_cp6(b):
    global _cp6_done, _transformer_results
    _cp6_out.clear_output(wait=True)
    with _cp6_out:
        from transformers import pipeline as hf_pipeline

        choice = _cp6_model_w.value
        n_sample = min(_cp6_sample_w.value, len(working_articles))
        session_data["cp6_model"] = choice

        if "DistilBERT" in choice:
            model_id = "distilbert-base-uncased-finetuned-sst-2-english"
            label_map = {"POSITIVE": "positive", "NEGATIVE": "negative"}
        else:
            model_id = "ProsusAI/finbert"
            label_map = {"positive": "positive", "negative": "negative", "neutral": "neutral"}

        print(f"Loading {model_id}...")
        pipe = hf_pipeline(
            "text-classification", model=model_id,
            truncation=True, max_length=128, device=-1,
        )

        sample_df = working_articles.sample(n=n_sample, random_state=42).copy()
        # Use headline + first 200 chars of article body for classification
        texts = (sample_df["headline"] + ". " + sample_df["text"].str[:200]).tolist()

        print(
            f"Running inference on {len(texts)} articles "
            "(this may take 1–3 minutes on CPU)..."
        )
        preds = pipe(texts, batch_size=8)

        sample_df["transformer_label"] = [
            label_map.get(p["label"].lower(), "neutral") for p in preds
        ]
        sample_df["transformer_score"] = [p["score"] for p in preds]

        # Store for CP7
        _transformer_results = sample_df.copy()

        # Compare transformer against VADER (which we already computed in CP4)
        vader = SentimentIntensityAnalyzer()
        sample_df["vader_compound"] = sample_df["text"].apply(
            lambda t: vader.polarity_scores(str(t)[:500])["compound"]
        )
        sample_df["vader_label"] = sample_df["vader_compound"].apply(
            lambda s: "positive" if s > 0.05 else ("negative" if s < -0.05 else "neutral")
        )

        # For DistilBERT (binary), collapse vader to binary too
        if "DistilBERT" in choice:
            sample_df["vader_label_cmp"] = sample_df["vader_label"].apply(
                lambda x: "positive" if x == "positive" else "negative"
            )
        else:
            sample_df["vader_label_cmp"] = sample_df["vader_label"]

        agreement = (sample_df["transformer_label"] == sample_df["vader_label_cmp"]).mean()
        session_data["cp6_finbert_accuracy"] = round(agreement, 4) if "FinBERT" in choice else None
        session_data["cp6_distilbert_accuracy"] = round(agreement, 4) if "DistilBERT" in choice else None

        _transformer_results = sample_df

        print(f"\nTransformer–VADER agreement rate: {agreement:.1%}")
        print("(Where they disagree, one method is likely making a domain or context error)\n")

        # Sentiment trajectory chart
        trans_yr = sample_df.groupby("year")["transformer_score"].mean()

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: transformer confidence score over time
        trans_yr.plot(
            ax=axes[0], marker="o", color="#6366f1",
            label=choice.split(" (")[0],
        )
        axes[0].axhline(0.5, color="grey", linestyle="--", linewidth=0.7)
        axes[0].set_title(
            "Transformer confidence score by year\n"
            "(>0.5 = confident in predicted class)"
        )
        axes[0].set_xlabel("Year")
        axes[0].set_ylabel("Mean confidence score")
        axes[0].legend(fontsize=8)

        # Right: stacked sentiment balance
        sent_dist = (
            sample_df.groupby(["year", "transformer_label"])
            .size()
            .unstack(fill_value=0)
        )
        sent_dist_pct = sent_dist.div(sent_dist.sum(axis=1), axis=0)
        color_map = {"positive": "#22c55e", "neutral": "#94a3b8", "negative": "#ef4444"}
        available_cols = [
            c for c in ["positive", "neutral", "negative"]
            if c in sent_dist_pct.columns
        ]
        sent_dist_pct[available_cols].plot(
            kind="bar", stacked=True, ax=axes[1],
            color=[color_map[c] for c in available_cols],
            edgecolor="white", linewidth=0.3,
        )
        axes[1].set_title(
            f"Transformer sentiment balance by year\n({choice.split('(')[0].strip()})"
        )
        axes[1].set_xlabel("Year")
        axes[1].tick_params(axis="x", rotation=45)
        axes[1].legend(title="Sentiment", fontsize=8)
        plt.tight_layout()
        plt.show()

        # Show a sample of disagreements — these are the most interesting cases
        disagree = sample_df[sample_df["transformer_label"] != sample_df["vader_label_cmp"]]
        if len(disagree) > 0:
            print(f"\nSample of VADER vs transformer disagreements ({len(disagree)} total):")
            print("These are the analytically interesting cases — investigate why they differ.\n")
            for _, row in disagree.head(5).iterrows():
                print(f"  Headline: {row['headline'][:80]}")
                print(f"  VADER: {row['vader_label_cmp']}  |  {choice.split('(')[0].strip()}: {row['transformer_label']}")
                print()

        _cp6_done = True
        print("\nCP6 complete. Run the CP6 Feedback cell, then proceed to CP7.")


_cp6_btn.on_click(_run_cp6)
display(widgets.VBox([_cp6_model_w, _cp6_sample_w, _cp6_btn, _cp6_out]))


In [ ]:
#@title CP6 — Feedback

validate_checkpoint(["_cp6_done"], "CP6")

_model_choice = session_data.get("cp6_model", "?")
display(Markdown(
    f"### CP6 Feedback — Fine-tuning, domain adaptation, and the accuracy–interpretability tradeoff\n\n"
    f"**Model choice: {_model_choice}**\n\n"
    "**What is fine-tuning?**  \n"
    "BERT was pre-trained on 3.3 billion words of Wikipedia and BookCorpus using two objectives: "
    "masked language modelling (predict a randomly hidden word) and next-sentence prediction. "
    "Pre-training teaches BERT to encode rich contextual representations of language — but *generic* language.\n\n"
    "Fine-tuning takes BERT's pre-trained weights and trains an additional classification layer on a "
    "labelled domain-specific dataset. FinBERT was fine-tuned on ~10,000 sentences from financial news "
    "and analyst reports, labelled positive/negative/neutral by domain experts. This shifts the model's "
    "internal representations to be more sensitive to financial vocabulary and framing.\n\n"
    "**Why VADER–transformer agreement matters:**  \n"
    "Without a hand-labelled ground-truth dataset for these specific articles, we cannot compute accuracy in "
    "the traditional sense. Instead, the agreement rate between VADER and your transformer is a diagnostic "
    "tool: high agreement means both methods are reading the same signal; disagreements reveal cases where "
    "context changes the interpretation. Disagreements are not errors — they are the most analytically "
    "interesting observations, because they expose the failure modes of each method.\n\n"
    "**Generic vs finance-tuned:**  \n"
    "- DistilBERT-SST2 was fine-tuned on movie review sentiment (Stanford Sentiment Treebank). The word "
    "*administration* in a film review has no financial meaning; the model is unlikely to weight it as "
    "strongly negative. In a BrewDog article, this is a critical failure.\n"
    "- FinBERT was exposed to *administration*, *restructuring*, *going concern* and similar terms in "
    "explicitly negative financial contexts. It assigns much stronger negative representations to these terms.\n\n"
    "**The interpretability cost:**  \n"
    "Unlike a dictionary method — where you can inspect exactly which words drove the score — a "
    "transformer's output is a function of 110 million parameters. You can visualise attention (as in CP5), "
    "but attention weights are not explanations: high attention on *losses* does not mean the model scored "
    "negatively *because* of *losses*; it means *losses* was contextually important, which is a weaker claim.\n\n"
    "> **Key principle for financial practice:** transformer models should be validated against human labels "
    "before deployment. The disagreement cases you saw above are exactly the instances that require human review."
))
print("Proceed to CP7.")


In [ ]:
#@title CP7 — Dictionary vs transformer: method comparison and interpretability

validate_checkpoint(["_cp6_done"], "CP6")

_cp7_interp_w = widgets.RadioButtons(
    options=[
        "Dictionary: I can see exactly which words drove the score (transparent but brittle)",
        "Transformer: I trust the model's contextual understanding (accurate but opaque)",
        "Neither: I would use both and flag discrepancies for human review",
    ],
    description="Your view on interpretability:",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "240px"},
)
_cp7_out = widgets.Output()
_cp7_btn = widgets.Button(
    description="Run comparison",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)


def _run_cp7(b):
    global _cp7_done
    _cp7_out.clear_output(wait=True)
    with _cp7_out:
        session_data["cp7_interpretability_choice"] = _cp7_interp_w.value

        df = _transformer_results.copy()

        # Ensure all three dictionary scores exist in df
        vader = SentimentIntensityAnalyzer()
        if "vader_compound" not in df.columns:
            df["vader_compound"] = df["text"].apply(
                lambda t: vader.polarity_scores(str(t)[:500])["compound"]
            )
        if "vader_label" not in df.columns:
            df["vader_label"] = df["vader_compound"].apply(
                lambda s: "positive" if s > 0.05 else ("negative" if s < -0.05 else "neutral")
            )

        # LM and Harvard IV-4 scores (re-use from CP4 if present, else compute)
        if "lm_score" not in df.columns:
            lm_neg = _load_wordlist(_LM_NEG_URL)
            lm_pos = _load_wordlist(_LM_POS_URL)
            def dict_score(tokens, pos_words, neg_words):
                p = sum(1 for t in tokens if t.lower() in pos_words)
                n = sum(1 for t in tokens if t.lower() in neg_words)
                return (p - n) / (p + n) if (p + n) > 0 else 0.0
            df["lm_score"] = df["tokens"].apply(lambda toks: dict_score(toks, lm_pos, lm_neg))

        # ── Agreement / disagreement analysis ──────────────────────────────
        is_binary = "DistilBERT" in session_data.get("cp6_model", "")
        if is_binary:
            df["vader_cmp"] = df["vader_label"].apply(
                lambda x: "positive" if x == "positive" else "negative"
            )
        else:
            df["vader_cmp"] = df["vader_label"]

        df["agree"] = df["transformer_label"] == df["vader_cmp"]
        agree_rate = df["agree"].mean()
        disagree = df[~df["agree"]].copy()

        model_short = session_data.get("cp6_model", "").split(" (")[0]

        print(f"Cross-method comparison on {len(df)}-article sample\n")
        print(f"  VADER–{model_short} agreement: {agree_rate:.1%}")
        print(f"  Disagreements: {len(disagree)}/{len(df)} articles ({len(disagree)/len(df):.0%})\n")

        # ── Sentiment trajectories: all methods side by side ───────────────
        yr_mean = df.groupby("year").agg(
            vader=("vader_compound", "mean"),
            transformer=("transformer_score", "mean"),
            lm=("lm_score", "mean") if "lm_score" in df.columns else ("transformer_score", "mean"),
        )

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: trajectory comparison
        axes[0].plot(yr_mean.index, yr_mean["vader"], marker="o",
                     color="#4f46e5", label="VADER", linewidth=1.8)
        axes[0].plot(yr_mean.index, yr_mean["transformer"], marker="s",
                     color="#10b981", label=model_short, linewidth=1.8, linestyle="--")
        if "lm_score" in df.columns:
            lm_yr = df.groupby("year")["lm_score"].mean()
            axes[0].plot(lm_yr.index, lm_yr.values, marker="^",
                         color="#f59e0b", label="LM Dictionary", linewidth=1.8, linestyle=":")
        axes[0].axhline(0, color="grey", linestyle="--", linewidth=0.7, alpha=0.6)
        axes[0].axvspan(
            2019, df["year"].max() + 0.5,
            alpha=0.06, color="red", zorder=0,
        )
        axes[0].set_title("Sentiment trajectory: all methods\n(red shading = controversy/crisis period)")
        axes[0].set_xlabel("Year")
        axes[0].set_ylabel("Mean score (direction comparable, not scale)")
        axes[0].legend(fontsize=8)

        # Right: confusion matrix style agreement heatmap (VADER vs transformer)
        all_labels = sorted(set(df["vader_cmp"].unique()) | set(df["transformer_label"].unique()))
        ct = pd.crosstab(df["vader_cmp"], df["transformer_label"],
                         rownames=["VADER"], colnames=[model_short])
        sns.heatmap(
            ct, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            cbar=False, linewidths=0.5,
        )
        axes[1].set_title(f"VADER vs {model_short}\nmethod agreement table (rows=VADER, cols=transformer)")
        plt.tight_layout()
        plt.show()

        # ── Deep dive: disagreement articles ──────────────────────────────
        print("\n── Disagreement deep-dive: where VADER and transformer diverge ──")
        print("These are the analytically richest cases — use them to identify failure modes.\n")
        if len(disagree) > 0:
            for _, row in disagree.head(5).iterrows():
                print(f"  Headline: {row['headline'][:75]}")
                print(f"  Year: {row['year']}  |  VADER: {row['vader_cmp']}  |  {model_short}: {row['transformer_label']}")
                vader_words = [
                    w for w in str(row["text"]).split()[:40]
                    if abs(vader.polarity_scores(w)["compound"]) > 0.2
                ]
                if vader_words:
                    print(f"  VADER-scoring tokens: {vader_words[:6]}")
                print(f"  For the transformer, no token-level explanation is available without SHAP/IG.")
                print()

        print("── Interpretability gap ──")
        print("Dictionary: you can audit exactly which words triggered the score.")
        print(f"Transformer: the score is a function of 110M parameters — you see only")
        print(f"  the output. This is the core interpretability tradeoff.\n")

        _cp7_done = True
        print("\nCP7 complete. Run the CP7 Feedback cell, then proceed to CP8.")


_cp7_btn.on_click(_run_cp7)
display(
    widgets.VBox(
        [
            widgets.HTML(
                "<b>Before running the comparison, record your view on interpretability:</b>"
            ),
            _cp7_interp_w,
            _cp7_btn,
            _cp7_out,
        ]
    )
)


In [ ]:
#@title CP7 — Feedback: ethical considerations in automated news sentiment analysis

validate_checkpoint(["_cp7_done"], "CP7")

display(Markdown(
    "### CP7 Feedback — Ethical considerations\n\n"
    "**1. Training data bias.**  \n"
    "Both VADER and FinBERT embed the biases of their training data. VADER was trained largely on "
    "English-language social media; FinBERT on English-language financial news. Neither model was exposed "
    "to non-English sources, non-Western financial contexts, or industry-specific language from sectors "
    "underrepresented in mainstream financial news. An analyst applying FinBERT to coverage of a Scottish "
    "craft brewery should recognise that the model's calibration may not reflect the specific framing "
    "conventions of UK tabloid journalism — which makes up the majority of this corpus (The Sun alone "
    "accounts for ~40% of articles). Tabloid coverage of BrewDog often focuses on scandal and celebrity; "
    "FinBERT, trained on FT-style financial reporting, may systematically misread this register.\n\n"
    "**2. Source diversity bias.**  \n"
    "This corpus draws from UK national newspapers with very different audiences and editorial stances. "
    "The Sun and Daily Mail reach mass-market readers and cover BrewDog primarily as a consumer brand. "
    "The Financial Times covers it as a listed-aspiring growth firm and then a distressed borrower. "
    "These are not the same BrewDog — they are different media constructions of it. Any aggregate "
    "sentiment score blends these distinct framings, which raises questions about what is actually "
    "being measured.\n\n"
    "**3. Temporal bias.**  \n"
    "Large language models have training cutoffs. A model trained before 2021 would not have encountered "
    "*'Equity for Punks'* in the specific negative framing it acquired after the open letter to James Watt. "
    "It would be interpreting that phrase with a positive prior, systematically underestimating the severity "
    "of reputational damage.\n\n"
    "**4. Automation bias.**  \n"
    "Research in human-computer interaction (Cummings, 2004; Goddard et al., 2012) demonstrates that humans "
    "systematically over-trust automated outputs, particularly when presented with a numerical score. A "
    "transformer sentiment score displayed to decimal places conveys false precision. In an investment "
    "analysis context, automated sentiment should be a screen, not a signal.\n\n"
    "**5. Explainability obligations.**  \n"
    "The EU AI Act (2024) classifies systems used for financial analysis as high-risk, requiring "
    "explainability of outputs. Dictionary methods satisfy explainability requirements directly (word "
    "matches are auditable); transformer outputs do not without supplementary SHAP or LIME analysis. "
    "This is not merely an academic concern — it has direct regulatory implications for firms "
    "deploying NLP in financial contexts.\n\n"
    "> **For your analyst note:** you are not expected to resolve these tensions. You are expected to name "
    "them, assess their likely direction of effect on your results, and state what additional validation "
    "you would require before acting on the outputs."
))
print("Proceed to CP8 — Analyst's note.")


## Analyst’s Note and Submission


In [ ]:
#@title CP8 — Analyst's note (150 words) and submission

validate_checkpoint(["_cp7_done"], "CP7")

display(Markdown(
    "### CP8 \u2014 Analyst's Note\n\n"
    "You have run two complementary analytical frameworks on the BrewDog news corpus:\n"
    "- **Topic modelling** (LDA) to identify what the coverage was *about*\n"
    "- **Dictionary sentiment** to measure the *tone* of coverage over time\n"
    "- **Transformer sentiment** to assess whether contextual understanding improves accuracy\n"
    "- **Head-to-head comparison** to evaluate the interpretability tradeoff\n\n"
    "Write a **maximum 150-word analyst's note** synthesising your findings. Your note should address:\n\n"
    "1. What the topic model reveals about how the narrative around BrewDog changed over time\n"
    "2. Whether dictionary or transformer sentiment better captured the crisis, and why\n"
    "3. At least one specific limitation that would affect the reliability of your conclusions\n"
    "4. What additional data or analysis you would require before submitting this as a formal "
    "investment or credit opinion\n\n"
    "Your note will be shared with the class in the debrief."
))

_cp8_note_w = widgets.Textarea(
    placeholder="Write your analyst note here (max 150 words)...",
    layout=widgets.Layout(width="700px", height="160px"),
)
_cp8_wc_label = widgets.Label("Word count: 0 / 150")
_cp8_out = widgets.Output()
_cp8_btn = widgets.Button(
    description="Submit note",
    button_style="success",
    layout=widgets.Layout(width="180px"),
)


def _update_wc(change):
    wc = len(change["new"].split())
    _cp8_wc_label.value = f"Word count: {wc} / 150"
    _cp8_wc_label.style.text_color = "#ef4444" if wc > 150 else "#22c55e"


_cp8_note_w.observe(_update_wc, names="value")


def _submit(b):
    _cp8_out.clear_output(wait=True)
    with _cp8_out:
        note = _cp8_note_w.value.strip()
        wc = len(note.split())
        if wc < 20:
            print("Please write at least 20 words before submitting.")
            return
        if wc > 165:
            print(f"Your note is {wc} words. Please reduce to 150 words.")
            return

        session_data["cp8_analyst_note"] = note
        session_data["timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())

        payload = {
            k: str(v) if v is not None else ""
            for k, v in session_data.items()
        }

        if not APPS_SCRIPT_URL:
            print(
                "APPS_SCRIPT_URL not set \u2014 submission skipped "
                "(instructor has not configured endpoint)."
            )
            print(f"\nYour session data:\n{json.dumps(payload, indent=2)}")
            return

        try:
            resp = requests.post(
                APPS_SCRIPT_URL, json=payload, timeout=20,
                headers={"Content-Type": "application/json"},
            )
            result = resp.json()
            if result.get("status") == "success":
                print(f"Submitted successfully \u2014 team: {session_data['team_name']}")
                print(f"Your analyst note ({wc} words):")
                print(f'\n"{note}"')
                print("\nWait for the instructor to display the class debrief.")
            else:
                print(f"Submission error: {result.get('message', 'unknown')}")
                print("Note saved locally. Contact your instructor if the problem persists.")
        except Exception as ex:
            print(f"Submission failed: {ex}")
            print("Note saved locally. Contact your instructor.")


_cp8_btn.on_click(_submit)
display(widgets.VBox([_cp8_note_w, _cp8_wc_label, _cp8_btn, _cp8_out]))
